# Notebook 12 – Model Selection

**Dataset:** Online Retail transactions (`data.csv`)

We compare models on two tasks:
- **Classification:** predict whether an order is from the UK (`Quantity`, `UnitPrice`, `TotalPrice` → `is_UK`)
- **Regression:** predict `UnitPrice` (`Quantity` → `UnitPrice`)

For each, we compare **Performance, Generalization, Training Time, Prediction Time, Complexity, Interpretability, and Resource Requirements**.

## Setup: Load Data

In [1]:
import pandas as pd, numpy as np, time
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(3000, random_state=42)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df.shape

(3000, 9)

# Part 1: Classification Models

Compare: **Logistic Regression, KNN, Decision Tree, Random Forest, SVM, Gradient Boosting**.

In [2]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
Xc = df[['Quantity', 'UnitPrice', 'TotalPrice']]
yc = (df['Country'] == 'United Kingdom').astype(int)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
clf_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN': KNeighborsClassifier(n_neighbors=10),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    'SVM': SVC(),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}
clf_rows = []
for name, model in clf_models.items():
    start = time.time()
    model.fit(Xc_train, yc_train)
    train_time = time.time() - start
    start = time.time()
    preds = model.predict(Xc_test)
    predict_time = time.time() - start
    train_acc = accuracy_score(yc_train, model.predict(Xc_train))
    test_acc = accuracy_score(yc_test, preds)
    cv_score = cross_val_score(model, Xc, yc, cv=5).mean()
    clf_rows.append({
        'Model': name,
        'Test Accuracy': round(test_acc, 3),
        'CV Score (Generalization)': round(cv_score, 3),
        'Train-Test Gap': round(train_acc - test_acc, 3),
        'Training Time (s)': round(train_time, 4),
        'Prediction Time (s)': round(predict_time, 5),
    })
clf_results = pd.DataFrame(clf_rows)
clf_results

,Model,Test Accuracy,CV Score (Generalization),Train-Test Gap,Training Time (s),Prediction Time (s)
0,Logistic Regression,0.897,0.895,-0.001,0.0356,0.00316
1,KNN,0.892,0.890,0.006,0.0079,0.00750
2,Decision Tree,0.890,0.892,0.014,0.0067,0.00260
3,Random Forest,0.897,0.895,0.008,0.5243,0.02827
4,SVM,0.897,0.897,0.001,0.1260,0.05796
5,Gradient Boosting,0.900,0.895,0.005,0.4024,0.00425


### Classification: Complexity, Interpretability & Resource Requirements
These aren't computed from code — they're properties of each algorithm.

In [9]:
clf_qualitative = pd.DataFrame({
    'Model': ['Logistic Regression', 'KNN', 'Decision Tree', 'Random Forest', 'SVM', 'Gradient Boosting'],
    'Complexity': ['Low', 'Low (but grows with data size)', 'Medium', 'High', 'Medium-High', 'High'],
    'Interpretability': ['High (coefficients)', 'Low', 'High (visualize tree)', 'Medium (feature importance)', 'Low', 'Low'],
    'Resource Requirements': ['Low', 'Low memory, but slow prediction on large data', 'Low', 'Medium (many trees)', 'Medium-High (scales poorly with rows)', 'Medium-High (sequential training)'],
})
clf_qualitative

,Model,Complexity,Interpretability,Resource Requirements
0,Logistic Regression,Low,High (coefficients),Low
1,KNN,Low (but grows with data size),Low,"Low memory, but slow prediction on large data"
2,Decision Tree,Medium,High (visualize tree),Low
3,Random Forest,High,Medium (feature importance),Medium (many trees)
4,SVM,Medium-High,Low,Medium-High (scales poorly with rows)
5,Gradient Boosting,High,Low,Medium-High (sequential training)


# Part 2: Regression Models

Compare: **Linear Regression, Ridge, Lasso, Decision Tree Regressor, Random Forest Regressor, Gradient Boosting Regressor**.

Task: predict `UnitPrice` from `Quantity`.

In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
Xr = df[['Quantity']]
yr = df['UnitPrice']
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)

In [11]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
reg_models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'Decision Tree Regressor': DecisionTreeRegressor(max_depth=6, random_state=42),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=100, random_state=42),
}
reg_rows = []
for name, model in reg_models.items():
    start = time.time()
    model.fit(Xr_train, yr_train)
    train_time = time.time() - start
    start = time.time()
    preds = model.predict(Xr_test)
    predict_time = time.time() - start
    train_preds = model.predict(Xr_train)
    train_r2 = r2_score(yr_train, train_preds)
    test_r2 = r2_score(yr_test, preds)
    cv_score = cross_val_score(model, Xr, yr, cv=5, scoring='r2').mean()
    reg_rows.append({
        'Model': name,
        'R² (Test)': round(test_r2, 3),
        'MAE': round(mean_absolute_error(yr_test, preds), 3),
        'RMSE': round(mean_squared_error(yr_test, preds) ** 0.5, 3),
        'CV R² (Generalization)': round(cv_score, 3),
        'Train-Test R² Gap': round(train_r2 - test_r2, 3),
        'Training Time (s)': round(train_time, 4),
        'Prediction Time (s)': round(predict_time, 5),
    })
reg_results = pd.DataFrame(reg_rows)
reg_results

,Model,R² (Test),MAE,RMSE,CV R² (Generalization),Train-Test R² Gap,Training Time (s),Prediction Time (s)
0,Linear Regression,-0.000,1.913,2.863,-0.005,0.003,0.0151,0.00234
1,Ridge,-0.000,1.913,2.863,-0.005,0.003,0.0032,0.00158
2,Lasso,-0.000,1.913,2.863,-0.005,0.003,0.0032,0.00125
3,Decision Tree Regressor,0.106,1.672,2.707,0.097,-0.039,0.0021,0.00115
4,Random Forest Regressor,0.106,1.670,2.707,0.098,-0.037,0.2068,0.01324
5,Gradient Boosting Regressor,0.107,1.668,2.706,0.099,-0.037,0.1178,0.00213


### Regression: Complexity, Interpretability & Resource Requirements

In [12]:
reg_qualitative = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge', 'Lasso', 'Decision Tree Regressor', 'Random Forest Regressor', 'Gradient Boosting Regressor'],
    'Complexity': ['Low', 'Low', 'Low (simpler, sparse)', 'Medium', 'High', 'High'],
    'Interpretability': ['High (coefficients)', 'High', 'High (some coefs = 0)', 'High (visualize tree)', 'Medium (feature importance)', 'Low'],
    'Resource Requirements': ['Very low', 'Very low', 'Very low', 'Low', 'Medium (many trees)', 'Medium-High (sequential training)'],
})
reg_qualitative

,Model,Complexity,Interpretability,Resource Requirements
0,Linear Regression,Low,High (coefficients),Very low
1,Ridge,Low,High,Very low
2,Lasso,"Low (simpler, sparse)",High (some coefs = 0),Very low
3,Decision Tree Regressor,Medium,High (visualize tree),Low
4,Random Forest Regressor,High,Medium (feature importance),Medium (many trees)
5,Gradient Boosting Regressor,High,Low,Medium-High (sequential training)
